# Lab 3 · Virtual flow metering (VFM)
**Slides:** Module 03 (production applications), 06 and 07  ·  **Time:** 75 min

A test separator measures oil, water and gas only occasionally. A **virtual flow meter** estimates the rates
continuously from gauges the well already has: choke size, bottom-hole and wellhead pressure, wellhead temperature,
separator pressure and temperature. The course notebook `VFM.ipynb` compares XGBoost, Extra Trees, CatBoost, random
forest and a neural network.

**You will:** find and fix a leakage problem · compare models with a group split · predict three phases · test extrapolation to an unseen choke · run a what-if.

In [ ]:
# --- setup: works locally and in Google Colab ---
import os, sys
if "google.colab" in sys.modules and not os.path.exists("aiml"):
    print("Upload the course kit zip to Colab, then unzip it:  !unzip -q aiml_course_kit.zip && %cd aiml_course_kit")
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("."))
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3, "axes.spines.top": False, "axes.spines.right": False})
DATA = "../data" if os.path.exists("../data") else "data"

from aiml.data import load_vfm, VFM_FEATURES, VFM_TARGETS
from aiml.models import regressors
from aiml.evaluate import compare, crossplot, report, importance
from sklearn.model_selection import train_test_split, GroupShuffleSplit, GroupKFold, KFold, cross_val_score
from sklearn.metrics import r2_score
df = load_vfm()
print(len(df), "rows ·", df.case_id.nunique(), "different well tests"); df.head()

## 1 · Why the course notebook scores R² ≈ 1.000
Each test appears once per separator pressure. Compare a random split with a split by `case_id`.

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor
X, y = df[VFM_FEATURES], df["oil_rate_stb_d"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=123)
et = ExtraTreesRegressor(n_estimators=300, random_state=0).fit(Xtr, ytr); report(yte, et.predict(Xte), "random split")
g = [c for c in VFM_FEATURES if c != "psep_psi"]
print("test rows whose well test is also in training (only Psep differs):", Xte[g].merge(Xtr[g].drop_duplicates(), how="inner").shape[0], "of", len(Xte))
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=1)
tr, te = next(gss.split(X, y, groups=df.case_id))
et.fit(X.iloc[tr], y.iloc[tr]); report(y.iloc[te], et.predict(X.iloc[te]), "split by case_id")

The drop is small here because the tests are dense, but the random-split number was not an honest estimate. From now on: **group split**.

In [ ]:
X_train, X_test, y_train, y_test = X.iloc[tr], X.iloc[te], y.iloc[tr], y.iloc[te]
keep = ["Linear regression", "KNN", "Decision tree", "Random forest", "Extra trees", "Gradient boosting", "Hist. gradient boosting (XGBoost-style)", "Neural network (MLP)"]
res = compare({k: v for k, v in regressors().items() if k in keep}, X_train, y_train, X_test, y_test)
res[["model", "R2_train", "R2_test", "MAE_test", "MAPE_%_test", "fit_s"]]

### ✏️ Your turn 1 · 10-fold cross-validation, done right
The notebook uses `KFold(10, shuffle=True)`. Repeat it for *Extra trees* with `KFold` and with `GroupKFold(10)` (groups = `case_id`). Which is optimistic?

In [ ]:
m = regressors()["Extra trees"]

## 2 · Cross-plots and residuals

In [ ]:
best = regressors()["Extra trees"].fit(X_train, y_train)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
crossplot(y_train, best.predict(X_train), y_test, best.predict(X_test), "Extra trees · oil rate", "STB/d", ax=ax[0])
ax[1].scatter(y_test, y_test - best.predict(X_test), s=12, alpha=.6); ax[1].axhline(0, color="r", ls="--")
ax[1].set_xlabel("actual oil rate"); ax[1].set_ylabel("residual"); ax[1].set_title("residuals (test)"); plt.tight_layout(); plt.show()

## 3 · Three phases
One model per phase. Which phase is hardest to meter virtually?

In [ ]:
rows = []
for t in VFM_TARGETS:
    m = regressors()["Extra trees"].fit(X_train, df[t].iloc[tr]); p = m.predict(X_test)
    rows.append({"target": t, **{k: round(v, 4) for k, v in report(df[t].iloc[te], p, t).items()}})
pd.DataFrame(rows)

## 4 · Extrapolation: a choke the model never saw
Hold out every test at one choke size and train on the other nine. Interpolating between chokes (0.70 in) is very different
from going beyond the data, and tree models cannot predict outside the range they saw.

In [ ]:
rows = []
for c in [0.25, 0.70, 1.94, 2.50]:
    m_tr = df.choke_in != c
    for name in ["Extra trees", "Hist. gradient boosting (XGBoost-style)", "Neural network (MLP)", "Linear regression"]:
        m = regressors()[name].fit(X[m_tr], y[m_tr])
        rows.append({"held-out choke (in)": c, "model": name, "R2": round(r2_score(y[~m_tr], m.predict(X[~m_tr])), 3)})
pd.DataFrame(rows).pivot(index="model", columns="held-out choke (in)", values="R2")

### ✏️ Your turn 2 · what-if
Keep a mid-life well (BHP 4700 psi, WHT 150 °F, Psep 300 psi, Tsep 100 °F) and vary the choke from 0.25 to 2.5 in.
Feed WHP from the test closest to each choke and plot the predicted oil rate. Why does the curve flatten above ~1.2 in?

In [ ]:
chokes = np.sort(df.choke_in.unique())

## 5 · Which gauges matter?

In [ ]:
importance(best, X_test, y_test); plt.title("permutation importance · oil rate"); plt.show()

### ✏️ Your turn 3 · a failed gauge
The wellhead pressure gauge fails. Retrain without `whp_psi` and compare the test R². Which other gauge takes over?

## Key ideas
* Duplicated rows make a random split leak: always ask "how many independent tests do I have?".
* VFM interpolates well; extrapolation to unseen operating conditions needs physics or new tests.
* On this data the six gauges predict all three phases well (test R² about 0.99 with a group split).
* An unseen choke size is the real weakness: the meter is reliable only inside the conditions it was trained on.
* Permutation importance shows which gauges the meter depends on, and what a failed gauge costs.

In [ ]:
# retrain without whp_psi